In [15]:
import torch
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, accuracy_score
import random
from sklearn.ensemble import VotingClassifier

# 读取数据
df_benign = pd.read_csv('benign_cyber.csv')
df_dos = pd.read_csv('dos_attack_cyber.csv')
df_replay = pd.read_csv('replay_attack_cyber.csv')
df_fdi = pd.read_csv('FDI_Cyber.csv')
df_evil_twin = pd.read_csv('Evil_Twin_Cyber.csv')
df_combined_3 = pd.concat([df_benign, df_dos, df_replay], ignore_index=False)
df_combined_2 = pd.concat([df_fdi,df_evil_twin],ignore_index=False)

features = [
    'wlan.fc.type', 'wlan.duration', 'ip.hdr_len', 'ip.dst',
    'frame.len', 'ip.proto', 'frame.protocols', 'wlan.seq', 'data.len',
    'time_since_last_packet', 'ip.src', 'ip.ttl', 'ip.id', 'wlan.ra',
    'llc.type', 'udp.dstport', 'class'  
]
# 提取 df_combined_3 和 df_combined_2 中的这些特征
df_combined_3_selected = df_combined_3[features]
df_combined_2_selected = df_combined_2[features]

KeyError: "['ip.hdr_len', 'ip.dst', 'ip.proto', 'frame.protocols', 'time_since_last_packet', 'ip.src', 'ip.ttl', 'ip.id', 'llc.type', 'udp.dstport'] not in index"

In [7]:

# 特征集：选择除最后四列之外的所有列
X = df_combined_3.iloc[:, :-1]

# 目标变量：选择倒数第四列
y = df_combined_3.iloc[:, -1]

# 分割数据为训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 特征缩放
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 神经网络模型
mlp1 = MLPClassifier(hidden_layer_sizes=(100,), max_iter=10, alpha=1e-4,
                    solver='adam', verbose=10, random_state=1,
                    learning_rate_init=.001)

# 训练模型
mlp1.fit(X_train_scaled, y_train)

# 预测
predictions = mlp1.predict(X_test_scaled)

# 性能评估
print(classification_report(y_test, predictions))
print(f"Accuracy: {accuracy_score(y_test, predictions)}")


Iteration 1, loss = 0.69623272
Iteration 2, loss = 0.46201340
Iteration 3, loss = 0.37254740
Iteration 4, loss = 0.32406089
Iteration 5, loss = 0.29111956
Iteration 6, loss = 0.26799403
Iteration 7, loss = 0.24999393
Iteration 8, loss = 0.23495764
Iteration 9, loss = 0.21862692
Iteration 10, loss = 0.20705239
              precision    recall  f1-score   support

  DoS attack       0.98      0.85      0.91      2360
      Replay       0.86      0.93      0.89      2394
      benign       0.93      0.99      0.96      1867

    accuracy                           0.92      6621
   macro avg       0.93      0.92      0.92      6621
weighted avg       0.92      0.92      0.92      6621

Accuracy: 0.9201027035191058


c:\Users\asus\.conda\envs\pytorch\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (10) reached and the optimization hasn't converged yet.
  warnings.warn(


In [8]:

# 特征集：选择除最后四列之外的所有列
X_2 = df_combined_2.iloc[:, :-4]

# 目标变量：选择倒数第四列
y_2 = df_combined_2.iloc[:, -4]

# 分割数据为训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 特征缩放
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 神经网络模型
mlp2 = MLPClassifier(hidden_layer_sizes=(100,), max_iter=10, alpha=1e-4,
                    solver='adam', verbose=10, random_state=1,
                    learning_rate_init=.001)

# 训练模型
mlp2.fit(X_train_scaled, y_train)

# 预测
predictions = mlp2.predict(X_test_scaled)

# 性能评估
print(classification_report(y_test, predictions))
print(f"Accuracy: {accuracy_score(y_test, predictions)}")


Iteration 1, loss = 0.69623272
Iteration 2, loss = 0.46201340
Iteration 3, loss = 0.37254740
Iteration 4, loss = 0.32406089
Iteration 5, loss = 0.29111956
Iteration 6, loss = 0.26799403
Iteration 7, loss = 0.24999393
Iteration 8, loss = 0.23495764
Iteration 9, loss = 0.21862692
Iteration 10, loss = 0.20705239
              precision    recall  f1-score   support

  DoS attack       0.98      0.85      0.91      2360
      Replay       0.86      0.93      0.89      2394
      benign       0.93      0.99      0.96      1867

    accuracy                           0.92      6621
   macro avg       0.93      0.92      0.92      6621
weighted avg       0.92      0.92      0.92      6621

Accuracy: 0.9201027035191058


c:\Users\asus\.conda\envs\pytorch\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (10) reached and the optimization hasn't converged yet.
  warnings.warn(


In [12]:
# 假设模型已经被训练并保存，现在加载它们
# 请替换以下代码中的 'mlp1.pkl' 和 'mlp2.pkl' 为你模型的文件路径
# mlp1 = pd.read_pickle('mlp1.pkl')
# mlp2 = pd.read_pickle('mlp2.pkl')

# 创建投票分类器
voting_clf = VotingClassifier(estimators=[
    ('mlp1', mlp1), 
    ('mlp2', mlp2)
], voting='hard')

# 假设投票分类器已经训练完毕，如果没有，你需要先使用 X_train 和 y_train 训练它
# voting_clf.fit(X_train_scaled, y_train)  # 只有在需要时才取消注释这行

# 随机选择一个数据集并随机抽取一条数据
random_dataset = random.choice(datasets)
random_sample = random_dataset.sample(n=1)

# 因为模型是在缩放过的数据上训练的，需要对抽取的样本进行相同的缩放处理
scaler = StandardScaler()
# 注意：此处应只使用transform，但因为我们没有fit过scaler，这里为了示例先使用fit_transform
random_sample_scaled = scaler.fit_transform(random_sample.drop('class', axis=1))  # 假设'class'是目标列

# 使用投票分类器进行预测
prediction = voting_clf.predict(random_sample_scaled)

# 打印预测结果
print("Predicted class:", prediction)


NotFittedError: This VotingClassifier instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.